In [1]:
import os

In [2]:
%pwd

'/home/cecil/projects/ares/notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/cecil/projects/ares'

In [5]:
import pandas as pd

In [6]:
data = pd.read_csv("artifacts/data/01-raw/raw.csv")

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15673 entries, 0 to 15672
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   url                  15673 non-null  object 
 1   fetch_date           15673 non-null  object 
 2   house_type           15673 non-null  object 
 3   bathrooms            15673 non-null  int64  
 4   bedrooms             15673 non-null  int64  
 5   price                15673 non-null  float64
 6   locality             15673 non-null  object 
 7   Condition            15630 non-null  object 
 8   Furnishing           15673 non-null  object 
 9   Property Size        15605 non-null  float64
 10  24-hour Electricity  15673 non-null  int64  
 11  Air Conditioning     15673 non-null  int64  
 12  Apartment            15673 non-null  int64  
 13  Balcony              15673 non-null  int64  
 14  Chandelier           15673 non-null  int64  
 15  Dining Area          15673 non-null 

In [8]:
data.isnull().sum()

url                     0
fetch_date              0
house_type              0
bathrooms               0
bedrooms                0
price                   0
locality                0
Condition              43
Furnishing              0
Property Size          68
24-hour Electricity     0
Air Conditioning        0
Apartment               0
Balcony                 0
Chandelier              0
Dining Area             0
Dishwasher              0
Hot Water               0
Kitchen Cabinets        0
Kitchen Shelf           0
Microwave               0
Pop Ceiling             0
Pre-Paid Meter          0
Refrigerator            0
TV                      0
Tiled Floor             0
Wardrobe                0
Wi-Fi                   0
loc                     0
dtype: int64

In [23]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    data_dir: Path
    all_schema: dict

In [24]:
from ares.constants import *
from ares.utils.common import read_yaml, create_directories

In [25]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            data_dir=config.data_dir,
            all_schema=schema,
        )
        return data_validation_config

In [26]:
import os
from ares import logger

In [ ]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = True

            data = pd.read_csv(self.config.data_dir)
            all_cols = list(data.columns)
            all_schema = dict(self.config.all_schema.items())

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"Validation status: {validation_status}")
                    return validation_status

            for col in all_cols:
                if data[col].dtype != all_schema[col]:
                    validation_status = False
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"Validation status: {validation_status}")
                    return validation_status

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [28]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-01-11 17:15:10,369: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-01-11 17:15:10,376: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-11 17:15:10,386: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-01-11 17:15:10,389: INFO: common: Created directory at: artifacts]
[2026-01-11 17:15:10,395: INFO: common: Created directory at: artifacts/data/data_validation]
